# Task 2
Implement basic backward pass using only numpy:
 - for your last week's Single Layer Perceptron.
 - for all activation functions
 - loss functions

Perform forward pass and backward pass, and use the gradient check function to verify your implementation...

In [1]:
import numpy as np
from utils import Module

## Linear Layer

In your previous task you defined `forward(input)` pass for your Linear class. Now we continue in creation of your own framework a little further with defining the `backward(dNet)` function. In this little framework is activation and linear unit separated. This separation is benefit in backward propagation and optimization. (If you want to know why, take a look on implementation of forward and backward propagation in class Model.)

Note, that now you are implementing backward pass for on only one sample, the lecture's framework was prepared for training on the whole dataset of $m$ samples.

In [2]:
#------------------------------------------------------------------------------
#   Linear class
#------------------------------------------------------------------------------
class Linear(Module):
    def __init__(self, in_features, out_features):
        super(Linear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = np.random.randn(out_features, in_features)
        self.dW = np.zeros_like(self.W)
        self.b = np.zeros((out_features, 1)) # Watch-out for the shape
        self.db = np.zeros_like(self.b)

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_inputs = input
        self.m = self.fw_inputs.shape[1]
        net = np.matmul(self.W, input) + self.b
        return net

    def backward(self, dz: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        self.dW = (1 / self.m) * np.matmul(dz, self.fw_inputs.T)
        self.db = (1 / self.m) * np.sum(dz, axis=1, keepdims=True)
        da_prev = np.matmul(self.W.T, dz)
        return da_prev
        # <<<<<<<<<

## Activations
Implement backward pass for Sigmoid, Tanh and ReLU activation functions.

In [3]:
#------------------------------------------------------------------------------
#   SigmoidActivationFunction class
#------------------------------------------------------------------------------
class Sigmoid(Module):
    def __init__(self):
        super(Sigmoid, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return 1.0 / (1.0 + np.exp(-input))

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        s = 1.0 / (1.0 + np.exp(-self.fw_input))
        return da * s * (1 - s)
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   HyperbolicTangentActivationFunction class
#------------------------------------------------------------------------------
class Tanh(Module):
    def __init__(self):
        super(Tanh, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return (np.exp(2 * input) - 1) / (np.exp(2 * input) + 1)

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        t = (np.exp(2 * self.fw_input) - 1) / (np.exp(2 * self.fw_input) + 1)
        return da * (1 - t ** 2)
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   RELUActivationFunction class
#------------------------------------------------------------------------------
class ReLU(Module):
    def __init__(self):
        super(ReLU, self).__init__()

    def forward(self, input: np.ndarray) -> np.ndarray:
        self.fw_input = input
        return np.maximum(input, 0)

    def backward(self, da) -> np.ndarray:
        # >>>>>>>>> add here
        return da * (self.fw_input > 0).astype(float)
        # <<<<<<<<<

## Loss functions
For successful backward pass, the computation and derivation of Loss function is necessary.
The most common Loss functions are **Mean Square Error** _(MSE, L2)_ **Mean Absolute Error** _(MAE, L1)_ and **Binary Cross Entropy** _(BCE, Log Loss)_ and their modifications according to what is better for the current dataset.

Implement SE and BCE Loss functions as Modules of our little framework.

Remember the difference between Loss and Cost.

In [4]:
#------------------------------------------------------------------------------
#   SquareErrorLossFunction class
#------------------------------------------------------------------------------
class SELoss(Module):
    def __init__(self):
        super(SELoss, self).__init__()

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        return (input - target) ** 2
        # <<<<<<<<<

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        return 2 * (input - target)
        # <<<<<<<<<

#------------------------------------------------------------------------------
#   BinaryCrossEntropyLossFunction class
#------------------------------------------------------------------------------
class BCELoss(Module):
    def __init__(self):
        super(BCELoss, self).__init__()

    def forward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        epsilon = 1e-12
        return -(target * np.log(input + epsilon) + (1 - target) * np.log(1 - input + epsilon))
        # <<<<<<<<<

    def backward(self, input: np.ndarray, target: np.ndarray) -> np.ndarray:
        # >>>>>>>>> add here
        epsilon = 1e-12
        return -(target / (input + epsilon) - (1 - target) / (1 - input + epsilon))
        # <<<<<<<<<

## Model
As in previous task, use `Model` class to encapsulate all layers of your MLP and define backward pass.
Iterate over its modules stored in parameter OrderedDict `modules` -> `self.modules` in the correct order.

Use call `.add_module(...)` to add layers of your MLP (network). Define MLP that could classify data from Circles dataset `dataset_Circles(...)`.


In [5]:
#------------------------------------------------------------------------------
#   Model class
#------------------------------------------------------------------------------
class Model(Module):
    def __init__(self):
        super(Model, self).__init__()

    def forward(self, input) -> np.ndarray:
        for name, module in self.modules.items():
            # print(f'Layer fw:{name}, a.shape = {input.shape} \n{input}')
            input = module(input)
            # print(f'z.shape = {input.shape} \n{input}')
        return input

    def backward(self, dz: np.ndarray):
        # >>>>>>>>> add here
        for name, module in reversed(self.modules.items()):
            dz = module.backward(dz)
        return dz
        # <<<<<<<<<

## Main Processing Cell

 1. Initialize dataset (`dataset_Circles`). [x]
 2. Declare a simple model (at least 3 hidden layer). [x]
 3. Perform forward pass through the network. [x]
 4. Compute loss. []
 5. Backward prop loss. []
 6. Backward pass MLP. []
 7. Check your computation of gradients via [`gradient_check`](https://datascience-enthusiast.com/DL/Improving_DeepNeural_Networks_Gradient_Checking.html) []
 8. Start crying. []
 9. Repeat until correct ;) []
 10. ... (if error founds -> blame lecturer) []

In [6]:
from dataset import dataset_Circles
from utils import gradient_check

In [7]:
dataset_features_X, dataset_labels_Y = dataset_Circles(m=128, radius=0.7, noise=0.0)

mlp = Model()
mlp.add_module(Linear(2, 3), 'Dense_1')
mlp.add_module(Tanh(), 'Tanh_1')
mlp.add_module(Linear(3, 4), 'Dense_2')
mlp.add_module(Tanh(), 'Tanh_2')
mlp.add_module(Linear(4, 5), 'Dense_3')
mlp.add_module(Tanh(), 'Tanh_3')
mlp.add_module(Linear(5, 1), 'Dense_4_out')
mlp.add_module(Sigmoid(), 'Sigmoid')

predicted_Y_hat = mlp.forward(dataset_features_X) # Be careful with the shape - Loss vs Cost

In [8]:
###>>> start of solution

loss_fn = BCELoss()

loss = loss_fn.forward(predicted_Y_hat, dataset_labels_Y)
cost = np.mean(loss)
print(f"Cost: {cost}")

dA = loss_fn.backward(predicted_Y_hat, dataset_labels_Y)
mlp.backward(dA)

###<<< end of solution

Cost: 0.7149349565297918


array([[-0.12997585,  0.11842566, -0.18025033, -0.03582874,  0.142557  ,
        -0.062277  , -0.09441309, -0.13713085, -0.09318521, -0.03127774,
         0.02840558, -0.14172839, -0.18928098, -0.01305725, -0.01018019,
        -0.30799653, -0.12793952,  0.07101645, -0.09380247, -0.25164162,
         0.01224761,  0.00453024, -0.16610668, -0.03352168, -0.02927305,
         0.02657782, -0.0138028 , -0.12237196,  0.08090357, -0.04104442,
         0.04927111,  0.1402408 ,  0.02275182, -0.13397226,  0.10704356,
        -0.29615134,  0.11950893, -0.11313608, -0.14612904, -0.13371731,
         0.12445237, -0.03581752, -0.06006875, -0.09905719, -0.28523231,
        -0.10036005, -0.0351266 ,  0.00881005,  0.01404931,  0.19361329,
         0.12994966,  0.04054863, -0.24266828,  0.17599308, -0.03742598,
         0.24878567,  0.20012414,  0.15789135,  0.07819918, -0.03176653,
        -0.0361307 , -0.29719947,  0.03525211, -0.27531945, -0.020572  ,
        -0.0428539 ,  0.19643538,  0.14473935,  0.1

In [9]:
# Verify your solution!
gradient_check(mlp, loss_fn, dataset_features_X, dataset_labels_Y)

Your backward propagation works perfectly fine! difference = 1.103925418424283e-08
